# Detection step-through inspector

Auto-discovers all labelling batches under . Select a batch and a candidate to step through the detection pipeline: ADSB ping overlay → cross-track gradient → rotated ROI mask → Canny edges → Hough lines.

In [ ]:
import sys, json, io, math
from pathlib import Path
import cv2, numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from concam.config import load_config
from concam.detection import _prepare_base
from concam.projection import PixelPoint, rotated_polygon

site_cfg = load_config(str(REPO_ROOT / "configs/mit_green_building.yaml"))
det_cfg  = site_cfg.detection
print(f"Config: preprocessing={det_cfg.preprocessing!r}  blur={det_cfg.blur_kernel}  ")
print(f"        angle_tol={det_cfg.angle_tolerance_deg}°  long_line_min={det_cfg.long_line_min_px}px")

In [ ]:
VAL_ROOT = REPO_ROOT / "output" / "validation" / "detection"

def discover_batches() -> list[str]:
    """Return sorted list of batch names that have a manifest.json."""
    if not VAL_ROOT.exists():
        return []
    return sorted(d.name for d in VAL_ROOT.iterdir()
                  if d.is_dir() and (d / "manifest.json").exists())

def load_batch(name: str) -> tuple[Path, list[dict]]:
    batch_dir = VAL_ROOT / name
    candidates = json.loads((batch_dir / "manifest.json").read_text())["candidates"]
    return batch_dir, candidates

def _img_widget(arr_bgr: np.ndarray, target_w: int = 370) -> widgets.Image:
    rgb = cv2.cvtColor(arr_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    new_h = max(1, int(h * target_w / w))
    rgb = cv2.resize(rgb, (target_w, new_h), interpolation=cv2.INTER_AREA)
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format="PNG")
    return widgets.Image(value=buf.getvalue(), format="png")

def _labeled(arr_bgr: np.ndarray, label: str, target_w: int = 370) -> widgets.VBox:
    return widgets.VBox([
        widgets.HTML(f"<div style='color:#9af;font-size:0.78rem;padding:2px 0'><b>{label}</b></div>"),
        _img_widget(arr_bgr, target_w),
    ])

batches = discover_batches()
print("Found batches:", batches)

In [ ]:
def build_panels(cand: dict, batch_dir: Path) -> dict | None:
    """
    Re-run the detection pipeline on the pre-extracted context crop and return
    six annotated panel images plus a metadata dict.

    Panel layout
    ─────────────────────────────────────────────────────
    1  Context + ADSB ping marker and flight-path arrow
    2  Context + rotated ROI polygon (orange) + AABB (thin blue)
    3  Cross-track gradient (_prepare_base output) over full context
    ─────────────────────────────────────────────────────
    4  Gradient zoomed to ROI area, polygon mask shaded green
    5  Canny edges within the polygon mask
    6  Hough lines overlaid on gradient
           green  = long-aligned  (≥ long_line_min_px AND within angle_tol)
           cyan   = short-aligned (< long_line_min_px but within angle_tol)
           gray   = angle-rejected
    """
    ctx_path = batch_dir / cand["context_png"]
    frame = cv2.imread(str(ctx_path))
    if frame is None:
        return None
    h_f, w_f = frame.shape[:2]

    px, py      = float(cand["pixel_x"]), float(cand["pixel_y"])
    path_vec    = (float(cand["path_dx"]), float(cand["path_dy"]))
    tl_x        = max(0, int(round(px)) - 400)   # context-crop top-left in full-frame coords
    tl_y        = max(0, int(round(py)) - 400)
    pin_x       = int(round(px)) - tl_x          # ADSB ping in crop coords
    pin_y       = int(round(py)) - tl_y

    # ── rotated polygon in crop coords ──────────────────────────────────────
    poly_ff   = rotated_polygon(PixelPoint(x=px, y=py), path_vec, det_cfg)
    poly_crop = poly_ff.copy().astype(np.float32)
    poly_crop[:, 0] -= tl_x
    poly_crop[:, 1] -= tl_y
    poly_int  = np.round(poly_crop).astype(np.int32)

    roi_d = cand["roi"]
    rx_c  = int(roi_d["x"]) - tl_x          # AABB top-left in crop coords
    ry_c  = int(roi_d["y"]) - tl_y

    # ── Panel 1: context + ADSB pin ────────────────────────────────────────
    p1 = frame.copy()
    al = 80
    cv2.circle(p1, (pin_x, pin_y), 12, (0, 220, 80), -1)
    cv2.circle(p1, (pin_x, pin_y), 12, (255, 255, 255), 2)
    cv2.circle(p1, (pin_x, pin_y), 4,  (255, 255, 255), -1)
    cv2.arrowedLine(p1, (pin_x, pin_y),
                    (int(round(pin_x + al * path_vec[0])),
                     int(round(pin_y + al * path_vec[1]))),
                    (0, 220, 80), 2, tipLength=0.25)
    sc_str  = f"  det={cand['detection_score']:.3f}" if isinstance(cand.get("detection_score"), float) else ""
    lbl_txt = f"{cand['callsign']}  {cand['wall_time_utc'][11:19]} UTC{sc_str}"
    cv2.putText(p1, lbl_txt, (pin_x + 16, pin_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 3, cv2.LINE_AA)
    cv2.putText(p1, lbl_txt, (pin_x + 16, pin_y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)

    # ── Panel 2: context + ROI polygon ─────────────────────────────────────
    p2 = frame.copy()
    cv2.polylines(p2, [poly_int], True, (0, 140, 255), 2)
    cv2.circle(p2, (pin_x, pin_y), 5, (0, 140, 255), -1)
    cv2.rectangle(p2, (rx_c, ry_c), (rx_c + int(roi_d["w"]), ry_c + int(roi_d["h"])),
                  (200, 140, 0), 1)

    # ── Panel 3: cross-track gradient (full context) ────────────────────────
    base, method_sfx = _prepare_base(frame, det_cfg, path_vec=path_vec)
    grad_u8 = cv2.normalize(base, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    p3 = cv2.cvtColor(grad_u8, cv2.COLOR_GRAY2BGR)
    cv2.polylines(p3, [poly_int], True, (0, 140, 255), 1)

    # ── ROI zoom for close-up panels ────────────────────────────────────────
    pad  = 30
    zx1  = max(0, rx_c - pad);  zy1 = max(0, ry_c - pad)
    zx2  = min(w_f, rx_c + int(roi_d["w"]) + pad)
    zy2  = min(h_f, ry_c + int(roi_d["h"]) + pad)
    z_base = base[zy1:zy2, zx1:zx2]
    zh, zw = z_base.shape[:2]

    lp = poly_crop.copy(); lp[:, 0] -= zx1; lp[:, 1] -= zy1
    lp_int = np.round(lp).astype(np.int32)
    zmask = np.zeros((zh, zw), dtype=np.uint8)
    cv2.fillPoly(zmask, [lp_int], 255)
    mvals = z_base[zmask > 0]

    # ── Panel 4: gradient ROI with mask ─────────────────────────────────────
    z_norm = cv2.normalize(z_base, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    p4 = cv2.cvtColor(z_norm, cv2.COLOR_GRAY2BGR)
    tint = np.zeros_like(p4); tint[zmask > 0] = (0, 50, 0)
    p4 = cv2.addWeighted(p4, 1.0, tint, 0.45, 0)
    cv2.polylines(p4, [lp_int], True, (0, 140, 255), 1)

    # ── Canny thresholds ─────────────────────────────────────────────────────
    if det_cfg.use_adaptive_canny and mvals.size > 0:
        p_hi = float(np.percentile(mvals, det_cfg.canny_percentile_high))
        c_hi = max(int(round(p_hi)), int(det_cfg.canny_min_high))
        c_lo = max(1, int(round(c_hi * det_cfg.canny_low_ratio)))
        floor_v = int(round(float(np.percentile(mvals, det_cfg.canny_percentile_low))))
    else:
        c_hi, c_lo, floor_v = int(det_cfg.canny_high), int(det_cfg.canny_low), 0

    crop_c = cv2.bitwise_and(z_base, z_base, mask=zmask)
    if floor_v > 0:
        _, crop_c = cv2.threshold(crop_c, floor_v, 255, cv2.THRESH_TOZERO)
    edges  = cv2.Canny(crop_c, c_lo, c_hi)
    edges  = cv2.bitwise_and(edges, edges, mask=zmask)

    # ── Panel 5: Canny edges ─────────────────────────────────────────────────
    p5 = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    cv2.polylines(p5, [lp_int], True, (0, 140, 255), 1)

    # ── Hough + angle filter ─────────────────────────────────────────────────
    lines_raw = cv2.HoughLinesP(
        edges, 1, np.pi / 180,
        threshold=int(det_cfg.hough_threshold),
        minLineLength=int(det_cfg.hough_min_line_length),
        maxLineGap=int(det_cfg.hough_max_line_gap),
    )
    path_ang = math.degrees(math.atan2(path_vec[1], path_vec[0])) % 180.0
    tol, min_long = float(det_cfg.angle_tolerance_deg), float(det_cfg.long_line_min_px)

    # ── Panel 6: Hough result ────────────────────────────────────────────────
    p6 = p4.copy()
    cv2.polylines(p6, [lp_int], True, (0, 140, 255), 1)
    n_al, n_long = 0, 0
    if lines_raw is not None:
        for ln in lines_raw:
            lx1, ly1, lx2, ly2 = ln[0]
            length   = math.hypot(lx2 - lx1, ly2 - ly1)
            la       = math.degrees(math.atan2(ly2 - ly1, lx2 - lx1)) % 180.0
            adiff    = abs(((la - path_ang + 90) % 180) - 90)
            if adiff <= tol:
                n_al += 1
                if length >= min_long:
                    n_long += 1
                    cv2.line(p6, (lx1,ly1),(lx2,ly2),(0,255,80),2)    # green = long-aligned
                else:
                    cv2.line(p6, (lx1,ly1),(lx2,ly2),(0,200,200),1)   # cyan  = short-aligned
            else:
                cv2.line(p6, (lx1,ly1),(lx2,ly2),(80,80,80),1)        # gray  = rejected

    meta = dict(
        callsign  = cand["callsign"],
        time      = cand["wall_time_utc"][11:19],
        det_score = cand.get("detection_score"),
        method    = "rotated_hough" + method_sfx,
        canny     = f"{c_lo}/{c_hi}",
        hough_thr = det_cfg.hough_threshold,
        n_aligned = n_al,
        n_long    = n_long,
        length_grown_px = cand.get("contrail_length_grown_px"),
        length_m        = cand.get("contrail_length_m"),
    )
    return dict(p1=p1, p2=p2, p3=p3, p4=p4, p5=p5, p6=p6, meta=meta)

print("build_panels() ready.")

In [ ]:
# ── Main UI ────────────────────────────────────────────────────────────────
if not batches:
    print("No labelling batches found under", VAL_ROOT)
else:
    batch_dd = widgets.Dropdown(
        options=batches, description="Batch:",
        layout=widgets.Layout(width="280px"),
    )
    cand_dd = widgets.Dropdown(
        options=[], description="Candidate:",
        layout=widgets.Layout(width="520px"),
    )
    out = widgets.Output()

    def _item(k, v):
        v_str = (f"{v:.3f}" if isinstance(v, float) else str(v)) if v is not None else "—"
        return f"<span style='color:#9af'>{k}:</span> <b>{v_str}</b>"

    def _refresh(_=None):
        val = cand_dd.value
        if val is None:
            return
        batch_dir, candidates = load_batch(batch_dd.value)
        cand = candidates[val]
        panels = build_panels(cand, batch_dir)
        if panels is None:
            with out:
                clear_output(); print("Image file not found.")
            return
        m = panels["meta"]
        score_col = "#4f4" if isinstance(m["det_score"], float) and m["det_score"] >= 0.3 else "#fa4"
        meta_row = " &nbsp;|&nbsp; ".join([
            _item("callsign",  m["callsign"]),
            _item("time",      m["time"] + " UTC"),
            f"<span style='color:#9af'>det_score:</span> <b style='color:{score_col}'>{m['det_score']:.3f}</b>" if isinstance(m["det_score"], float) else _item("det_score", m["det_score"]),
            _item("method",    m["method"]),
            _item("canny lo/hi", m["canny"]),
            _item("n_aligned", m["n_aligned"]),
            f"<span style='color:#9af'>n_long_aligned:</span> <b style='color:#4f4'>{m['n_long']}</b>",
            _item("length_grown_px", m["length_grown_px"]),
            _item("length_m",        f"{m['length_m']:.0f} m" if isinstance(m["length_m"], float) else m["length_m"]),
        ])
        W, WZ = 360, 320
        row1 = widgets.HBox([
            _labeled(panels["p1"], "① Context + ADSB ping", W),
            _labeled(panels["p2"], "② Rotated ROI polygon (orange) + AABB (thin)", W),
            _labeled(panels["p3"], f"③ Cross-track gradient ({m['method']})", W),
        ], layout=widgets.Layout(gap="10px"))
        row2 = widgets.HBox([
            _labeled(panels["p4"], "④ Gradient ROI — masked to polygon", WZ),
            _labeled(panels["p5"], f"⑤ Canny edges (lo/hi = {m['canny']})", WZ),
            _labeled(panels["p6"], "⑥ Hough lines: green=long-aligned  cyan=short-aligned  gray=rejected", WZ),
        ], layout=widgets.Layout(gap="10px"))
        with out:
            clear_output(wait=True)
            display(widgets.VBox([
                widgets.HTML(f"<div style='font-size:0.8rem;line-height:2;padding:4px 0'>{meta_row}</div>"),
                row1,
                widgets.HTML("<div style='height:8px'></div>"),
                row2,
            ]))

    def _on_batch(change):
        if change["name"] != "value": return
        batch_dir, candidates = load_batch(change["new"])
        def _opt(i, c):
            s = f"#{i:02d}  {c['callsign']:<10} {c['wall_time_utc'][11:19]}"
            if isinstance(c.get("detection_score"), float):
                s += f"  det={c['detection_score']:.3f}"
            return (s, i)
        cand_dd.options = [_opt(i, c) for i, c in enumerate(candidates)]

    def _on_cand(change):
        if change["name"] != "value": return
        _refresh()

    batch_dd.observe(_on_batch, names="value")
    cand_dd.observe(_on_cand,  names="value")

    _on_batch({"name": "value", "new": batch_dd.value})   # initial load

    display(widgets.VBox([
        widgets.HBox([batch_dd, cand_dd]),
        out,
    ]))